In [1]:
# Analyzing the full dataset of 81 grid points and generating MSD and D figures for report

In [2]:
# required installations amongst others

#!pip install git+https://gitlab.tue.nl/20235660/fftmsd.git
#!pip install git+https://github.com/lhillma/Cell2Image.git

In [3]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from fftmsd import calc_msd_fft, unwrap, calc_acf
from pathlib import Path
from cell2image.image import read_h5_lattice_snapshots
from cell2image.shape import (
    get_perimeter_estimator,
    calc_area,
    calc_shape_index,)
import networkx as nx
import matplotlib.pyplot as plt

plt.rcParams['xtick.direction'] = plt.rcParams['ytick.direction'] = 'in' # Plot all ticks inside

In [4]:
# START WITH COMPUTATIONS INVOLVING COM: msd, d

In [ ]:
# Define the file paths to the simulation output of the 5x5 grid

file_paths = [
    f"scratch/Simulation_batch3/Simulation{i}/com.h5"
    for i in range(1,301)]

print(len(file_paths))

In [5]:
# Define the file paths to the simulation output of the expansion of the grid to 9x9

file_paths_GE = [
    f"scratch/Simulation_batch3_GE/Simulation{i}/com.h5"
    for i in range(1,673)]
file_paths_GE_MC = [
    f"scratch/Simulation_batch3_GE_mistakeCorrection/Simulation{i}/com.h5"
    for i in range(1,49)]

file_paths_GE[24:36]   = file_paths_GE_MC[0:12]
file_paths_GE[84:96]   = file_paths_GE_MC[12:24]
file_paths_GE[144:156] = file_paths_GE_MC[24:36]
file_paths_GE[204:216] = file_paths_GE_MC[36:48]

print(len(file_paths_GE))

672


In [6]:
# Generate the MSD's and save the file (wrong way)

# with h5py.File("msd_batch3_GE.h5", "w") as f_out:
#     for i, fp in enumerate(file_paths_GE):
#         with h5py.File(fp, "r") as f:
#             data = f["simulation/cluster_com"][:, :, 0]   
#             msd_i= calc_msd_fft(data)

#         f_out.create_dataset(f"msd_{i}", data=msd_i)

In [ ]:
# Generate the MSD's and save the file (correct way)

with h5py.File("msd_batch3_averaged.h5", "w") as f_out:
    for i, fp in enumerate(file_paths):
        with h5py.File(fp, "r") as f:
            data = f["simulation/cluster_com"][:, :]
            msd_i= np.mean([calc_msd_fft(data[:, i]) for i in range(100)],axis=0)

        f_out.create_dataset(f"msd_{i}", data=msd_i)

In [ ]:
# Generate the MSD's and save the file (correct way)

with h5py.File("msd_batch3_GE_averaged[corrected].h5", "w") as f_out:
    for i, fp in enumerate(file_paths_GE):
        with h5py.File(fp, "r") as f:
            data = f["simulation/cluster_com"][:, :]
            msd_i= np.mean([calc_msd_fft(data[:, i]) for i in range(100)],axis=0)

        f_out.create_dataset(f"msd_{i}", data=msd_i)

In [5]:
# Plot the MSD on a loglog scale every 12 simulations (so from every grid point one)

msd = []
with h5py.File("msd_batch3_averaged.h5", "r") as f:
    for i in range(len(f.keys())):
        msd.append(f[f"msd_{i}"][:])

msd_GE = []
with h5py.File("msd_batch3_GE_averaged[corrected].h5", "r") as f:
    for i in range(len(f.keys())):
        msd_GE.append(f[f"msd_{i}"][:])

plt.figure()

for i in range(0, 300, 12):
    plt.plot(msd[i], label=f"Simul{i+1}")
for i in range(0, 672, 12):
    plt.plot(msd_GE[i], label=f"Simul{i+1}")

plt.xscale('log')
plt.yscale('log')
plt.ylim(1e1, 1e8)
plt.xlim(1e0, 1e5)
plt.xlabel("lag time")
plt.ylabel("MSD")

plt.savefig("DataAnalysis_GE_MC/msd_GE_every12_001.png", dpi=200)
plt.close()

In [6]:
# Plot average MSD and standard deviation per different setting pair

msd = np.array(msd)   
msd_GE = np.array(msd_GE)   

grouped_old = msd.reshape(25, 12, -1)   # 25 different settings and 12 simulations per setting
grouped_GE = msd_GE.reshape(56, 12, -1)   # 56 different settings and 12 simulations per setting

grouped = np.concatenate((grouped_old, grouped_GE))

mean_msd = grouped.mean(axis=1)
std_msd  = grouped.std(axis=1)

plt.figure()

for i in range(81):
    time_step = 100
    t = np.arange(len(mean_msd[i])) * time_step

    plt.plot(t, mean_msd[i], label=f"Settings {i+1}")
    plt.fill_between(
        t,
        mean_msd[i] - std_msd[i],
        mean_msd[i] + std_msd[i],
        alpha=0.15
    )

plt.xscale('log')
plt.yscale('log')
plt.ylim(1e2, 1e8)
plt.xlim(1e2, 1e7)
plt.xlabel("time [MCS]")
plt.ylabel(r"MSD [$px^2$]")
plt.savefig("DataAnalysis_GE_MC/msd_AvgStd_GE_001.png", dpi=200)
plt.close()

In [8]:
# Function to compute diffusion constants from the MSD trajectories on the real MCS time steps

def compute_diffusion_from_msd(msd, time_step, d, time_boundary=1e5):
    """
    msd: individual msd trajectories
    time_step: the time interval in between generation of data of com (in this case 100)
    d: dimension of the problem (in this case d=2)
    time_boundary: time value that is the boundary for calculating the diffusion constant
    """
    t = np.arange(len(msd)) * time_step
    mask = t >= time_boundary
    slope = np.polyfit(t[mask], msd[mask], 1)[0]
    D = slope / (2 * d) 
    return D, t

In [9]:
# Compute diffusion constants

D = []
msd_tot = np.concatenate((msd, msd_GE))

for msd_i in msd_tot:
    D.append(compute_diffusion_from_msd(msd_i, 100, 2)[0])

In [25]:
# Plot the average diffusion constant grid such that the ticks are in the middle of the boxes

D = np.array(D)
D_avg = D.reshape(-1, 12).mean(axis=1)

# custom ordering of indices
order = [
    1,46,2,47,3,48,4,49,5,
    26,66,27,67,28,68,29,69,30,
    6,50,7,51,8,52,9,53,10,
    31,70,32,71,33,72,34,73,35,
    11,54,12,55,13,56,14,57,15,
    36,74,37,75,38,76,39,77,40,
    16,58,17,59,18,60,19,61,20,
    41,78,42,79,43,80,44,81,45,
    21,62,22,63,23,64,24,65,25]

# convert to zero-based indexing
order = np.array(order) - 1

D_grid = D_avg[order]
grid = D_grid.reshape((9, 9), order='F')
grid = np.flipud(grid)

# parameter values (cell centers)
phi = np.linspace(0.30, 0.70, 9)
fact = np.linspace(4, 12, 9)

# cell spacings
dphi = phi[1] - phi[0]      # 0.05
dfact = fact[1] - fact[0]   # 1.0

# custom image boundaries (cell edges)
extent = [
    phi[0] - dphi/2,     # 0.275
    phi[-1] + dphi/2,    # 0.725
    fact[0] - dfact/2,   # 3.5
    fact[-1] + dfact/2]  # 12.5


plt.figure(figsize=(6, 5))

im = plt.imshow(
    grid,
    extent=extent,
    aspect='auto',
    interpolation='none')

plt.xlabel(r'$\phi$')
plt.ylabel(r'$F_{act}$')

# ticks at cell centers
plt.xticks(phi)
plt.yticks(fact)

plt.colorbar(im, label='Diffusion constant')

plt.tight_layout()

plt.savefig(
    "DataAnalysis_GE_MC/Diffusion_gridGE_004.png",
    dpi=200,
    bbox_inches="tight")

plt.close()

In [ ]:
# Print the averaged diffusion constants in grid order

print(grid)

In [30]:
# plot histogram of D

plt.figure()
plt.hist(D_grid, bins=6)
plt.xlabel("Diffusion constant")
plt.ylabel("Counts")
plt.xlim(0, 6)
plt.savefig("DataAnalysis_GE_MC/histogramD_batch3_GE_006.png", dpi=200)
plt.close()

[[5.55915748 5.78924909 5.0023114  4.37477813 3.80893207 3.18800255
  2.23227672 1.38949804 0.4688214 ]
 [4.37832473 4.23336085 3.83406965 3.39753386 2.89484675 2.11603059
  1.56498412 0.9288108  0.12721232]
 [3.08107049 2.79487348 2.71145078 2.38534509 1.90408688 1.44914605
  0.99258405 0.41671922 0.0514271 ]
 [2.16771038 1.99484928 1.54156825 1.38150479 1.26239724 0.95321355
  0.48927987 0.09819579 0.05166901]
 [1.33104888 1.17389575 1.15992293 0.88914629 0.63937349 0.48344151
  0.11011053 0.02716196 0.02348848]
 [0.63484597 0.48796555 0.49384581 0.41113356 0.22451887 0.15459851
  0.04506558 0.05128548 0.01819696]
 [0.11080212 0.1370112  0.08545106 0.16097079 0.06377891 0.09393891
  0.03683947 0.0277012  0.01227036]
 [0.11007016 0.06430398 0.07521798 0.09029089 0.06544694 0.03260319
  0.02992679 0.01398006 0.03342218]
 [0.04729241 0.02108617 0.02574927 0.0476419  0.04140798 0.01619806
  0.02846515 0.0117491  0.03560344]]


In [14]:
# Plot the MSD average trajectories on color scale of diffusion constants

t = np.arange(mean_msd.shape[1]) * 100

# colormap based on diffusion constants
norm = mpl.colors.Normalize(vmin=D_avg.min(), vmax=D_avg.max())
cmap = plt.cm.viridis
sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

fig, ax = plt.subplots()

for i in range(81):
    color = cmap(norm(D_avg[i]))

    ax.plot(t, mean_msd[i], color=color)

    ax.fill_between(
        t,
        mean_msd[i] - std_msd[i],
        mean_msd[i] + std_msd[i],
        color=color,
        alpha=0.15)

fig.colorbar(sm, ax=ax, label="Diffusion constant")

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_ylim(1e1, 1e9)
ax.set_xlim(1e2, 1e7)

ax.set_xlabel("time [MCS]")
ax.set_ylabel(r"MSD [$px^2$]")

plt.savefig("DataAnalysis_GE_MC/msdStd_batch3GE_004.png", dpi=200)
plt.close()

(81,)


In [15]:
# Plot for every phi the diffusion constant against the active force

F_act = np.array([4, 5, 6, 7, 8, 9, 10, 11, 12])

# reshape diffusion constants into (phi, F_act)
D_grid = D_grid.reshape(9, 9)

phi_vals = np.linspace(0.30, 0.70, 9)

fig, ax = plt.subplots()

for i, phi in enumerate(phi_vals):
    ax.plot(F_act, D_grid[i], marker='o', label=rf'$\phi={phi:.2f}$')

ax.set_xlabel(r'$F_{act}$')
ax.set_ylabel('Diffusion constant')

ax.legend()
plt.savefig("DataAnalysis_GE_MC/DvsFact_perPhi_batch3GE_001.png", dpi=200)
plt.close()

In [16]:
# FROM HERE ON COMPUTATIONS WITH LATTICE: area, perimeter, shape index, nucleus area fraction

In [ ]:
# Obtain evidence to include nucleus area fraction as graph feature

In [126]:
# Get the lattice frame at t = 50.000 MCS for a single simulation

frame_list = read_h5_lattice_snapshots(Path("scratch/Simulation_batch3_GE/Simulation350/lattice.h5"))
frame = frame_list[600] 

# Include the perimeter estimator 
perimeter_estimator = get_perimeter_estimator()

In [127]:
# Create lists of areas, perimeters and shape indices of nucleus for one single snapshot  

field_nuc = frame.cell_id  

# Measure nucleus by the cell id (has odd id's)
nucleus_areas = []
nucleus_perimeters = []
nucleus_shape_indices = [] 

for cell_id in range(200):
    if cell_id %2 ==1:
        nucleus_areas.append(calc_area(field_nuc, cell_id))
        nucleus_perimeters.append(perimeter_estimator(field_nuc, cell_id))
        nucleus_shape_indices.append(calc_shape_index(field_nuc, cell_id, perimeter_estimator))

print(f"Area: {nucleus_areas[0]}, Perimeter: {nucleus_perimeters[0]:.2f}, Shape index: {nucleus_shape_indices[0]:.3f}")

Area: 446, Perimeter: 76.22, Shape index: 3.609


In [128]:
# Create lists of areas, perimeters and shape indices of cell for one single snapshot  

field_cell = frame.cluster_id  

# Measure cell by the cluster id (so cell + nucleus area)
cell_areas = []
cell_perimeters = []
cell_shape_indices = [] 

for cell_id in range(100):
    cell_areas.append(calc_area(field_cell, cell_id))
    cell_perimeters.append(perimeter_estimator(field_cell, cell_id))
    cell_shape_indices.append(calc_shape_index(field_cell, cell_id, perimeter_estimator))

print(f"Area: {cell_areas[0]}, Perimeter: {cell_perimeters[0]:.2f}, Shape index: {cell_shape_indices[0]:.3f}")

Area: 899, Perimeter: 112.17, Shape index: 3.741


In [129]:
# create nucleus area fraction

cellareas = np.array(cell_areas)
nucleusareas = np.array(nucleus_areas)

fraction_allcells = nucleusareas / cellareas
fraction_global = np.mean(fraction_allcells)

print(round(fraction_global, 4))

0.4986


In [130]:
# Visualize nucleus and cell areas

print(min(cell_areas))
print(max(cell_areas))
print(min(nucleus_areas))
print(max(nucleus_areas))

894
907
443
457


In [131]:
# Get the minimal and maximal nucleus area fraction within the snapshot

print(len(fraction_allcells))
xmax = fraction_allcells.max()
xmin = fraction_allcells.min()
print(f"The maximal area fraction is:", xmax)
print(f"The minimal area fraction is:", xmin)

100
The maximal area fraction is: 0.503858875413451
The minimal area fraction is: 0.49333333333333335


In [132]:
# create histogram of area fraction variations across cells in the same snapshot

plt.figure()

bins = np.linspace(xmin, xmax, 21)  # 20 bins
plt.hist(fraction_allcells, bins=bins, range=(xmin, xmax), edgecolor='black')
plt.xlim(xmin - 0.0005, xmax + 0.0005)

plt.xlabel(r"Nucleus area fraction ($\phi$)")
plt.ylabel("Counts")
plt.savefig("DataAnalysis_GE/hist_AreaFrac_report_007.png", dpi=200)
plt.close()